<a href="https://colab.research.google.com/github/aninhanascimento/aninhanascimento/blob/main/Entra21_PLN_2026_2_Aula_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introdução ao PLN: preparação de textos e similaridade

Nesta atividade, utilizaremos um pequeno conjunto de **livros fictícios e suas sinopses** para explorar algumas operações fundamentais do Processamento de Linguagem Natural (PLN).

Nosso objetivo principal será investigar uma questão simples:

> **Como diferentes formas de preparar e representar um texto afetam a maneira como calculamos sua similaridade com outros textos?**

Ao longo da atividade, partiremos de representações bastante simples e acrescentaremos gradualmente novas técnicas de processamento.

Inicialmente, vamos:

1. carregar e explorar nosso corpus de sinopses;
2. realizar uma **tokenização simples** dos textos;
3. comparar documentos utilizando a **similaridade de Jaccard**;
4. representar os documentos utilizando **Bag of Words (BoW)**;
5. calcular a **similaridade de cosseno** entre seus vetores.

Em seguida, investigaremos experimentalmente o efeito de diferentes técnicas de preparação dos textos, como:

- conversão para **lowercase**;
- remoção de **pontuação**;
- remoção de **stopwords**;
- aplicação de **stemming**.

A ideia é modificar progressivamente o pipeline e observar como cada transformação interfere nos valores de similaridade obtidos.

---

### Uma ideia importante

Em PLN, o texto original não é comparado diretamente. Antes disso, precisamos tomar uma série de decisões sobre como transformá-lo em uma representação que possa ser processada computacionalmente.

Podemos pensar no processo geral como:

**Texto original → preparação → representação → medida de similaridade → resultado**

Diferentes escolhas em cada uma dessas etapas podem produzir resultados diferentes.

Por isso, ao longo desta atividade, não estaremos procurando apenas o **maior valor de similaridade**, mas tentando compreender **por que os resultados mudam e se essas mudanças tornam a comparação entre os textos mais adequada para nosso objetivo**.

## 1. Carregando nosso corpus de livros

Nesta atividade, utilizaremos um pequeno corpus contendo **50 livros fictícios**, divididos em quatro gêneros:

- Young Adult;
- Ficção científica;
- Autoajuda;
- Computação.

Para cada livro, temos informações como título, autor, gênero e uma **sinopse**. A sinopse será nosso principal dado textual.

Antes de aplicar qualquer técnica de Processamento de Linguagem Natural, é importante conhecer a estrutura dos dados e verificar como os textos estão armazenados.

In [ ]:
import pandas as pd
from google.colab import files

# Faz o upload do arquivo
uploaded = files.upload()

# Recupera o nome do arquivo enviado
nome_arquivo = next(iter(uploaded))

# Carrega a planilha
df = pd.read_excel(nome_arquivo, sheet_name="dados_livros")

print(f"Número de livros: {len(df)}")
print(f"Número de colunas: {len(df.columns)}")

df

Saving livros_sinteticos_pln_ptbr.xlsx to livros_sinteticos_pln_ptbr.xlsx
Número de livros: 50
Número de colunas: 7


,ID,genero,autor,titulo,nr_pags,editora,sinopse
0,LIV001,Young Adult,Lívia Montenegro,As Chaves de Aurória: A Cidade sob Vidro,336,Editora Horizonte Violeta,"Aos dezessete anos, Nina descobre que a cidade..."
1,LIV002,Young Adult,Lívia Montenegro,As Chaves de Aurória: O Arquivo das Sombras,352,Editora Horizonte Violeta,Depois de romper parte do controle sobre Aurór...
2,LIV003,Young Adult,Lívia Montenegro,As Chaves de Aurória: Depois da Cúpula,368,Editora Horizonte Violeta,"Com a cúpula de Aurória finalmente aberta, Nin..."
3,LIV004,Young Adult,Caio Nogueira,Verão em Quase Lugar Nenhum,288,Editora Maré de Papel,Theo esperava passar as férias jogando videoga...
4,LIV005,Young Adult,Marina Salgado,Playlist para um Coração em Reforma,304,Editora Maré de Papel,Depois de terminar um namoro às vésperas do úl...
5,LIV006,Young Adult,Rafael Ishikawa,Clube dos Invisíveis,320,Editora Ponto de Virada,Cinco estudantes que raramente conversam são c...
6,LIV007,Young Adult,Beatriz Farias,Sete Dias para Mudar de Escola,272,Editora Ponto de Virada,Luna recebe a notícia de que sua família vai s...
7,LIV008,Young Adult,Júlia Ferraz,O Mapa das Coisas que Não Dissemos,312,Editora Horizonte Violeta,"Após a morte do avô, Clara encontra um mapa de..."
8,LIV009,Young Adult,Andréa Luz,Entre Provas e Constelações,296,Editora Neblina Jovem,Maya é conhecida pelas melhores notas da turma...
9,LIV010,Young Adult,Samuel Porto,Manual para Sobreviver ao Último Ano,264,Editora Neblina Jovem,"No início do terceiro ano, quatro amigos escre..."


### Explorando os dados

Vamos verificar os tipos das variáveis, a distribuição dos gêneros e observar algumas sinopses.

Neste momento, não faremos nenhuma transformação nos textos: queremos primeiro trabalhar com os dados em sua forma original.

In [ ]:
# Estrutura do DataFrame
df.info()

print("\nLivros por gênero:")
display(df["genero"].value_counts())

# Exemplo de alguns livros e suas sinopses
display(
    df[["ID", "genero", "titulo", "sinopse"]].head(5)
)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 7 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   ID       50 non-null     object
 1   genero   50 non-null     object
 2   autor    50 non-null     object
 3   titulo   50 non-null     object
 4   nr_pags  50 non-null     int64 
 5   editora  50 non-null     object
 6   sinopse  50 non-null     object
dtypes: int64(1), object(6)
memory usage: 2.9+ KB

Livros por gênero:


,count
genero,
Young Adult,13
Ficção científica,13
Autoajuda,12
Computação,12


,ID,genero,titulo,sinopse
0,LIV001,Young Adult,As Chaves de Aurória: A Cidade sob Vidro,"Aos dezessete anos, Nina descobre que a cidade..."
1,LIV002,Young Adult,As Chaves de Aurória: O Arquivo das Sombras,Depois de romper parte do controle sobre Aurór...
2,LIV003,Young Adult,As Chaves de Aurória: Depois da Cúpula,"Com a cúpula de Aurória finalmente aberta, Nin..."
3,LIV004,Young Adult,Verão em Quase Lugar Nenhum,Theo esperava passar as férias jogando videoga...
4,LIV005,Young Adult,Playlist para um Coração em Reforma,Depois de terminar um namoro às vésperas do úl...


### Função auxiliar

Como iremos comparar diferentes livros várias vezes, criaremos uma pequena função para visualizar um registro a partir de seu ID.

In [ ]:
def mostrar_livro(id_livro):
    livro = df.loc[df["ID"] == id_livro].iloc[0]

    print(f"ID:     {livro['ID']}")
    print(f"Título: {livro['titulo']}")
    print(f"Gênero: {livro['genero']}")
    print()
    print(livro["sinopse"])


mostrar_livro("LIV001")

ID:     LIV001
Título: As Chaves de Aurória: A Cidade sob Vidro
Gênero: Young Adult

Aos dezessete anos, Nina descobre que a cidade de Aurória é protegida por uma cúpula de vidro que também apaga lembranças consideradas perigosas. Quando seu melhor amigo desaparece dos registros escolares, ela encontra uma chave metálica escondida no antigo observatório e passa a investigar a origem do sistema. Entre provas finais, amizades abaladas e uma paixão inesperada, Nina precisa decidir se revela um segredo capaz de libertar os jovens da cidade ou se preserva a segurança de sua família. Aventura, identidade, amizade, rebeldia e descoberta marcam o primeiro volume da trilogia.


## 2. Tokenização: transformando texto em palavras

Um texto é inicialmente apenas uma sequência de caracteres.

Uma das operações mais básicas em PLN é a **tokenização**, isto é, dividir o texto em unidades menores chamadas *tokens*. Neste exemplo, trataremos aproximadamente cada palavra como um token.

Por enquanto, nossa tokenização será propositalmente simples:

- manteremos maiúsculas e minúsculas;
- manteremos acentos;
- não removeremos stopwords;
- não aplicaremos stemming ou lematização.

Nosso objetivo é estabelecer uma primeira referência (*baseline*) antes de realizar preparações mais sofisticadas.

In [ ]:
import re

def tokenizar(texto):
    """
    Tokenização simples baseada em palavras.
    """
    return re.findall(r"\b\w+\b", texto)


texto_exemplo = df.loc[df["ID"] == "LIV001", "sinopse"].iloc[0]

tokens = tokenizar(texto_exemplo)

print(tokens[:30])
print()
print(f"Número total de tokens: {len(tokens)}")

['Aos', 'dezessete', 'anos', 'Nina', 'descobre', 'que', 'a', 'cidade', 'de', 'Aurória', 'é', 'protegida', 'por', 'uma', 'cúpula', 'de', 'vidro', 'que', 'também', 'apaga', 'lembranças', 'consideradas', 'perigosas', 'Quando', 'seu', 'melhor', 'amigo', 'desaparece', 'dos', 'registros']

Número total de tokens: 91


### Similaridade de Jaccard

Uma forma bastante simples de comparar dois textos é observar quantas palavras eles possuem em comum.

Para isso, podemos representar cada documento como um **conjunto de tokens**.

A similaridade de Jaccard é definida como:

$$
J(A,B) =
\frac{\lvert A \cap B \rvert}
{\lvert A \cup B \rvert}
$$

onde:

- $A \cap B$ representa as palavras presentes nos dois textos;
- $A \cup B$ representa todas as palavras diferentes presentes nos dois textos.

O resultado varia entre **0 e 1**:

- `0` → nenhum token em comum;
- `1` → os dois conjuntos de tokens são idênticos.

Como utilizamos conjuntos, a frequência das palavras não é considerada.

In [ ]:
def similaridade_jaccard(texto1, texto2):
    tokens1 = set(tokenizar(texto1))
    tokens2 = set(tokenizar(texto2))

    intersecao = tokens1.intersection(tokens2)
    uniao = tokens1.union(tokens2)

    return len(intersecao) / len(uniao)


# Dois volumes da mesma série
texto1 = df.loc[df["ID"] == "LIV001", "sinopse"].iloc[0]
texto2 = df.loc[df["ID"] == "LIV002", "sinopse"].iloc[0]

sim = similaridade_jaccard(texto1, texto2)

print(f"Similaridade de Jaccard: {sim:.3f}")

Similaridade de Jaccard: 0.133


### Comparando diferentes pares

Como nosso corpus foi construído com livros de diferentes gêneros e também com sequências de uma mesma série, podemos comparar pares que intuitivamente deveriam apresentar diferentes níveis de similaridade.

Vamos comparar:

1. dois volumes da mesma série;
2. um dos volumes com um livro de outro gênero.

In [ ]:
def comparar_jaccard(id1, id2):
    livro1 = df.loc[df["ID"] == id1].iloc[0]
    livro2 = df.loc[df["ID"] == id2].iloc[0]

    sim = similaridade_jaccard(
        livro1["sinopse"],
        livro2["sinopse"]
    )

    print(f"{livro1['titulo']}")
    print("versus")
    print(f"{livro2['titulo']}")
    print(f"\nJaccard = {sim:.3f}")


# Livros da mesma série
comparar_jaccard("LIV001", "LIV002")

print("\n" + "-" * 60 + "\n")

# Livros bastante diferentes
comparar_jaccard("LIV001", "LIV049")

As Chaves de Aurória: A Cidade sob Vidro
versus
As Chaves de Aurória: O Arquivo das Sombras

Jaccard = 0.133

------------------------------------------------------------

As Chaves de Aurória: A Cidade sob Vidro
versus
Fundamentos de Cibersegurança

Jaccard = 0.040


## 3. Bag of Words

Na análise anterior, cada documento foi representado apenas pelo conjunto de palavras que continha. Assim, perdemos uma informação importante: **quantas vezes cada palavra aparece**.

O modelo **Bag of Words (BoW)** cria um vocabulário contendo os tokens encontrados no corpus e representa cada documento por meio de suas frequências.

Um exemplo simplificado:

| documento | cidade | escola | amizade |
|---|---:|---:|---:|
| Documento A | 2 | 1 | 3 |
| Documento B | 1 | 2 | 1 |

Cada linha pode ser interpretada como um **vetor numérico**.

A ordem das palavras é ignorada — daí a ideia de uma "sacola de palavras" —, mas suas frequências são preservadas.

Por enquanto, continuaremos utilizando nossa tokenização simples, sem lowercase, remoção de stopwords ou stemming.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(
    tokenizer=tokenizar,
    token_pattern=None,
    lowercase=False
)

# Cria a matriz documento-termo
bow = vectorizer.fit_transform(df["sinopse"])

print("Dimensões da matriz Bag of Words:")
print(bow.shape)

print()
print("Número de documentos:", bow.shape[0])
print("Tamanho do vocabulário:", bow.shape[1])

Dimensões da matriz Bag of Words:
(50, 1608)

Número de documentos: 50
Tamanho do vocabulário: 1608


### Observando a representação

A matriz completa possui muitas palavras. Para compreender melhor sua estrutura, podemos visualizar uma pequena parte dela.

Cada linha corresponde a um livro e cada coluna corresponde a uma palavra do vocabulário.

In [ ]:
# Converte apenas uma pequena parte da matriz para visualização
bow_df = pd.DataFrame(
    bow[:5].toarray(),
    columns=vectorizer.get_feature_names_out()
)

# Seleciona somente palavras que aparecem nesses cinco documentos
colunas_usadas = bow_df.columns[bow_df.sum() > 0]

display(
    bow_df[colunas_usadas]
    .iloc[:, :20]
)

,A,Aos,Aurória,Aventura,Com,Depois,Elisa,Enquanto,Entre,Joana,Jovens,Lá,Miguel,Nina,No,O,Quando,Ravi,Romance,Theo
0,0,1,1,1,0,0,0,0,1,0,0,0,0,2,0,0,1,0,0,0
1,1,0,1,0,0,1,0,0,0,0,0,0,0,2,0,2,0,0,0,0
2,0,0,1,0,1,0,0,0,0,0,1,0,0,2,1,0,0,0,0,0
3,1,0,0,0,0,0,0,0,1,1,0,1,0,0,0,0,0,1,0,2
4,0,0,0,0,0,1,1,1,0,0,0,0,1,0,0,1,0,0,1,0


### Similaridade de cosseno

Depois de representar cada documento como um vetor, podemos comparar a direção desses vetores utilizando a **similaridade de cosseno**.

A similaridade entre dois vetores $A$ e $B$ é calculada por:

$$
\text{cosine}(A,B) =
\frac{A \cdot B}
{\lVert A \rVert \lVert B \rVert}
$$

onde:

- $A \cdot B$ representa o **produto escalar** entre os vetores;
- $\lVert A \rVert$ representa o **comprimento (norma)** do vetor $A$;
- $\lVert B \rVert$ representa o **comprimento (norma)** do vetor $B$.

Intuitivamente, dois documentos terão alta similaridade quando seus vetores apontarem para direções semelhantes, isto é, quando apresentarem padrões semelhantes de uso das palavras.

Para vetores **Bag of Words**, os valores normalmente variam entre:

- próximo de **0** → documentos pouco semelhantes;
- próximo de **1** → documentos muito semelhantes.

Diferentemente do Jaccard, agora estamos comparando **vetores de frequências**, e não apenas conjuntos de palavras.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Calcula a similaridade entre todos os documentos
matriz_similaridade = cosine_similarity(bow)

print("Dimensões da matriz de similaridade:")
print(matriz_similaridade.shape)

Dimensões da matriz de similaridade:
(50, 50)


### Comparando os mesmos livros novamente

Vamos utilizar os mesmos pares da análise com Jaccard.

Isso nos permite observar como uma mudança na **representação dos documentos** e na **medida de similaridade** modifica os resultados.

In [ ]:
def similaridade_cosseno(id1, id2):
    i = df.index[df["ID"] == id1][0]
    j = df.index[df["ID"] == id2][0]

    return matriz_similaridade[i, j]


pares = [
    ("LIV001", "LIV002"),
    ("LIV001", "LIV049")
]

for id1, id2 in pares:

    livro1 = df.loc[df["ID"] == id1].iloc[0]
    livro2 = df.loc[df["ID"] == id2].iloc[0]

    jac = similaridade_jaccard(
        livro1["sinopse"],
        livro2["sinopse"]
    )

    cos = similaridade_cosseno(id1, id2)

    print(f"{id1} × {id2}")
    print(f"{livro1['titulo']}")
    print(f"{livro2['titulo']}")
    print(f"Jaccard: {jac:.3f}")
    print(f"Cosseno + BoW: {cos:.3f}")
    print("-" * 60)

LIV001 × LIV002
As Chaves de Aurória: A Cidade sob Vidro
As Chaves de Aurória: O Arquivo das Sombras
Jaccard: 0.133
Cosseno + BoW: 0.472
------------------------------------------------------------
LIV001 × LIV049
As Chaves de Aurória: A Cidade sob Vidro
Fundamentos de Cibersegurança
Jaccard: 0.040
Cosseno + BoW: 0.381
------------------------------------------------------------


## Até aqui: o que mudou?

Utilizamos o mesmo corpus de documentos de duas maneiras diferentes:

**Tokens + Jaccard**

`texto → tokens → conjunto de palavras → Jaccard`

**Bag of Words + cosseno**

`texto → tokens → vetor de frequências → similaridade de cosseno`

Ainda não realizamos praticamente nenhuma limpeza dos textos.

A partir daqui, investigaremos uma nova questão:

> **O que acontece com os resultados de similaridade quando modificamos progressivamente a preparação dos textos?**

Testaremos operações como lowercase, remoção de stopwords e stemming, mantendo controladas as demais etapas do experimento.



---

## Exercício 1 — Como a preparação do texto afeta a similaridade?

Até aqui, calculamos a similaridade entre as sinopses utilizando uma representação **Bag of Words (BoW)** e a **similaridade de cosseno**, com pouca preparação prévia dos textos.

Agora, vamos investigar como diferentes operações de pré-processamento podem alterar os resultados.

Crie versões progressivamente preparadas das sinopses, aplicando as seguintes etapas:

1. **Texto original** — sem novas transformações;
2. **Lowercase** — converter todas as palavras para letras minúsculas;
3. **Remoção de pontuação**;
4. **Remoção de stopwords** da língua portuguesa;
5. **Stemming** — reduzir palavras a uma forma radical aproximada.

A cada nova etapa:

- mantenha também todas as transformações realizadas anteriormente;
- construa uma nova representação Bag of Words;
- calcule novamente a similaridade de cosseno;
- compare os resultados para os mesmos pares de livros utilizados anteriormente.

Registre os resultados em uma tabela semelhante a esta:

| Preparação | LIV001 × LIV002 | LIV001 × LIV049 |
|---|---:|---:|
| Original | ... | ... |
| + lowercase | ... | ... |
| + remoção de pontuação | ... | ... |
| + remoção de stopwords | ... | ... |
| + stemming | ... | ... |

Ao final, observe:

- A similaridade aumentou ou diminuiu em cada etapa?
- As transformações afetaram igualmente pares semelhantes e pares diferentes?
- Qual operação parece ter provocado a maior mudança?
- Um valor maior de similaridade significa necessariamente um resultado melhor?

In [ ]:
# ============================================================
# Bibliotecas que podem ser úteis no exercício
# ============================================================

import re

import nltk
from nltk.corpus import stopwords
from nltk.stem import RSLPStemmer

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# Baixa os recursos do NLTK necessários para português
nltk.download("stopwords")
nltk.download("rslp")


# ============================================================
# Texto fictício apenas para demonstrar as sintaxes
# Não utilize este exemplo como solução do exercício.
# ============================================================

texto_exemplo = "Os Programadores estavam criando novos Sistemas, rapidamente!"


# ------------------------------------------------------------
# 1. LOWERCASE
# Converte todos os caracteres para letras minúsculas
# ------------------------------------------------------------

texto_minusculo = texto_exemplo.lower()

print(texto_minusculo)


# ------------------------------------------------------------
# 2. REMOÇÃO DE PONTUAÇÃO
# Mantém apenas caracteres de palavras e espaços
# ------------------------------------------------------------

texto_sem_pontuacao = re.sub(
    r"[^\w\s]",
    "",
    texto_exemplo
)

print(texto_sem_pontuacao)


# ------------------------------------------------------------
# 3. TOKENIZAÇÃO SIMPLES
# Divide o texto em palavras
# ------------------------------------------------------------

tokens = re.findall(
    r"\b\w+\b",
    texto_exemplo
)

print(tokens)


# ------------------------------------------------------------
# 4. STOPWORDS
# Carrega palavras muito frequentes da língua portuguesa,
# como "de", "a", "o", "para", "uma" etc.
# ------------------------------------------------------------

stopwords_pt = set(stopwords.words("portuguese"))

tokens_sem_stopwords = [
    token
    for token in tokens
    if token.lower() not in stopwords_pt
]

print(tokens_sem_stopwords)


# ------------------------------------------------------------
# 5. STEMMING
# O RSLPStemmer foi desenvolvido para a língua portuguesa.
# Ele reduz diferentes formas de uma palavra a um radical.
# ------------------------------------------------------------

stemmer = RSLPStemmer()

tokens_stem = [
    stemmer.stem(token)
    for token in tokens
]

print(tokens_stem)


# ------------------------------------------------------------
# 6. RECONSTRUÇÃO DO TEXTO
# Depois de trabalhar com uma lista de tokens,
# podemos reuni-los novamente em uma string.
# ------------------------------------------------------------

texto_processado = " ".join(tokens_sem_stopwords)

print(texto_processado)


# ------------------------------------------------------------
# 7. BAG OF WORDS
# fit_transform() aprende o vocabulário e transforma
# uma coleção de textos em vetores de frequências.
# ------------------------------------------------------------

textos_exemplo = [
    "dados linguagem programação",
    "programação algoritmos dados"
]

vectorizer = CountVectorizer()

matriz_bow = vectorizer.fit_transform(textos_exemplo)

print(vectorizer.get_feature_names_out())
print(matriz_bow.toarray())


# ------------------------------------------------------------
# 8. SIMILARIDADE DE COSSENO
# Compara os vetores produzidos pelo Bag of Words.
# ------------------------------------------------------------

similaridades = cosine_similarity(matriz_bow)

print(similaridades)

os programadores estavam criando novos sistemas, rapidamente!
Os Programadores estavam criando novos Sistemas rapidamente
['Os', 'Programadores', 'estavam', 'criando', 'novos', 'Sistemas', 'rapidamente']
['Programadores', 'criando', 'novos', 'Sistemas', 'rapidamente']
['os', 'program', 'est', 'cri', 'nov', 'sistem', 'rapid']
Programadores criando novos Sistemas rapidamente
['algoritmos' 'dados' 'linguagem' 'programação']
[[0 1 1 1]
 [1 1 0 1]]
[[1.         0.66666667]
 [0.66666667 1.        ]]


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package rslp to /root/nltk_data...
[nltk_data]   Unzipping stemmers/rslp.zip.


## Exercício 2 — Busca e recomendação por similaridade textual

No exercício anterior, você investigou como diferentes formas de preparação dos textos afetam a representação **Bag of Words** e os valores de **similaridade de cosseno**.

Agora, utilize esse mesmo princípio para construir duas aplicações simples:

1. um **mecanismo de busca textual**;
2. um **sistema de recomendação de livros**.

### Parte A — Mecanismo de busca

Implemente uma função que receba uma consulta textual (*query*) escrita pelo usuário e retorne os **5 livros mais similares** à consulta.

Exemplo:

`buscar("aventura juvenil envolvendo amizade e mistério")`

O mecanismo deverá seguir, em linhas gerais, o fluxo:

**query → preparação → Bag of Words → similaridade de cosseno → Top 5**

Para isso:

- aplique à query a **mesma preparação textual** utilizada nas sinopses;
- transforme a query utilizando o **mesmo `CountVectorizer` ajustado ao corpus**;
- calcule a similaridade de cosseno entre a query e todas as sinopses;
- ordene os resultados da maior para a menor similaridade;
- apresente os **5 livros mais similares**.

O resultado deve informar, pelo menos:

| posição | ID | título | gênero | similaridade |
|---|---|---|---|---:|

Teste o mecanismo utilizando pelo menos **três consultas diferentes** e observe quais tipos de consulta produzem resultados mais coerentes.

### Parte B — Sistema de recomendação

Implemente uma função que receba o **ID de um livro** e retorne os **5 livros mais similares** a ele.

Exemplo:

`recomendar("LIV001")`

Nesse caso, o próprio vetor Bag of Words do livro escolhido será comparado com os vetores dos demais livros.

O fluxo pode ser representado como:

**livro → vetor do livro → similaridade com os demais → Top 5**

Sua função deverá:

- localizar o livro correspondente ao ID informado;
- recuperar sua representação Bag of Words;
- comparar esse vetor com os demais livros;
- excluir o próprio livro dos resultados;
- ordenar os resultados pela similaridade;
- retornar os **5 livros mais similares**.

Apresente os resultados no mesmo formato utilizado no mecanismo de busca.

### Para refletir

Depois de implementar os dois sistemas, responda brevemente:

- Qual é a principal diferença entre **busca** e **recomendação** neste exemplo?
- Os livros mais similares pertencem necessariamente ao mesmo gênero?
- Como a preparação textual escolhida pode modificar os resultados?
- Que limitações você identifica em um sistema baseado apenas em **Bag of Words**?
- O que acontece quando uma consulta utiliza palavras relacionadas ao conteúdo dos livros, mas que não fazem parte do vocabulário aprendido pelo modelo?

> **Desafio:** compare os resultados utilizando duas formas diferentes de preparação textual desenvolvidas no Exercício 1. Os resultados da busca ou as recomendações mudam? Qual configuração parece produzir resultados mais coerentes?